# Greeks: analytical vs. finite-difference

Checks on `pricing.greeks`:

1. **Delta and Gamma vs. spot**: plot the closed-form Black-Scholes Greeks across a range of spot prices.
2. **Analytical vs. finite difference**: overlay a central finite-difference estimate on top of the analytical curve, for both Delta (first derivative) and Gamma (second derivative).
3. **Why finite-difference Gamma is noisier**: sweep the finite-difference step size `h` for Gamma and show the classic truncation-error vs. rounding-error trade-off.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pricing.black_scholes import black_scholes_price
from pricing.greeks import analytical_greeks, finite_difference_derivative

In [ ]:
# Market parameters (same as the Black-Scholes and Monte Carlo notebooks, for comparability)
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20
q = 0.02

greeks_call = analytical_greeks(S0, K, T, r, sigma, option_type="call", q=q)
greeks_put = analytical_greeks(S0, K, T, r, sigma, option_type="put", q=q)
print(f"Call Greeks: {greeks_call}")
print(f"Put Greeks:  {greeks_put}")

## 1. Delta and Gamma vs. spot

In [ ]:
spot_grid = np.linspace(50, 150, 200)

delta_call = [analytical_greeks(S, K, T, r, sigma, option_type="call", q=q).delta for S in spot_grid]
delta_put = [analytical_greeks(S, K, T, r, sigma, option_type="put", q=q).delta for S in spot_grid]
gamma_call = [analytical_greeks(S, K, T, r, sigma, option_type="call", q=q).gamma for S in spot_grid]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(spot_grid, delta_call, label="Call delta")
axes[0].plot(spot_grid, delta_put, label="Put delta")
axes[0].axvline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
axes[0].set_xlabel("Spot price S")
axes[0].set_ylabel("Delta")
axes[0].set_title("Delta vs. spot")
axes[0].legend()

axes[1].plot(spot_grid, gamma_call, label="Gamma (call = put)", color="tab:green")
axes[1].axvline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
axes[1].set_xlabel("Spot price S")
axes[1].set_ylabel("Gamma")
axes[1].set_title("Gamma vs. spot")
axes[1].legend()

fig.tight_layout()
plt.show()

## 2. Analytical vs. finite difference (overlay)

In [ ]:
fd_delta_call = []
fd_gamma_call = []
for S in spot_grid:
    base_kwargs = dict(S=S, K=K, T=T, r=r, sigma=sigma, option_type="call", q=q)
    fd_delta_call.append(
        finite_difference_derivative(black_scholes_price, base_kwargs, param_name="S", h=0.01, order=1)
    )
    fd_gamma_call.append(
        finite_difference_derivative(black_scholes_price, base_kwargs, param_name="S", h=0.5, order=2)
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(spot_grid, delta_call, label="Analytical delta", linewidth=2)
axes[0].plot(spot_grid, fd_delta_call, "--", label="Finite-difference delta")
axes[0].axvline(K, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Spot price S")
axes[0].set_ylabel("Delta")
axes[0].set_title("Delta: analytical vs. finite difference")
axes[0].legend()

axes[1].plot(spot_grid, gamma_call, label="Analytical gamma", linewidth=2, color="tab:green")
axes[1].plot(spot_grid, fd_gamma_call, "--", label="Finite-difference gamma", color="tab:red")
axes[1].axvline(K, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Spot price S")
axes[1].set_ylabel("Gamma")
axes[1].set_title("Gamma: analytical vs. finite difference")
axes[1].legend()

fig.tight_layout()
plt.show()

## 3. Why finite-difference Gamma is noisier: sweeping the step size `h`

Gamma's finite-difference formula divides by `h**2`. For large `h` the estimate is biased by truncation error (the curve is not exactly quadratic over a wide step). For very small `h`, floating-point rounding error in the three subtracted prices gets amplified by the `1/h**2` factor, and the estimate becomes noisy rather than more accurate. Delta only divides by `2*h` (not squared), so it does not suffer this as badly.

In [ ]:
h_values = np.logspace(-5, 1, 60)  # from 1e-5 (too small) to 10 (too large)
base_kwargs = dict(S=S0, K=K, T=T, r=r, sigma=sigma, option_type="call", q=q)
true_gamma = greeks_call.gamma

gamma_errors = [
    abs(
        finite_difference_derivative(black_scholes_price, base_kwargs, param_name="S", h=h, order=2)
        - true_gamma
    )
    for h in h_values
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(h_values, gamma_errors, "o-")
ax.set_xlabel("Finite-difference step size h")
ax.set_ylabel("|FD gamma - analytical gamma|")
ax.set_title("Gamma finite-difference error vs. step size h")
fig.tight_layout()
plt.show()